# Generate Stage 4 Kaggle submissions

This notebook is self-contained. It creates missing Stage 2 and Stage 3 training artifacts, predicts every test row with the honest Stage 4 pipeline, validates exact sample-submission IDs and order, and saves primary Stage 4 and secondary Stage 3 CSV files to Google Drive. Stage 5 spatial correction is intentionally excluded because it failed its promotion gates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import pandas as pd

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
data_dir = Path('/content/rogii-data')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
drive_data = drive_root / 'data'
assert (drive_data / 'train').is_dir(), f'Missing train directory: {drive_data / "train"}'
assert (drive_data / 'test').is_dir(), f'Missing test directory: {drive_data / "test"}'
assert (drive_data / 'sample_submission.csv').is_file(), 'Missing sample_submission.csv'
if not ((data_dir / 'train').is_dir() and (data_dir / 'test').is_dir() and (data_dir / 'sample_submission.csv').is_file()):
    shutil.copytree(drive_data, data_dir, dirs_exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
stage2_dir = artifact_dir / 'stage2_pf_trend_blend_full_v001'
if not (stage2_dir / 'metrics.json').is_file():
    assert not stage2_dir.exists() or not any(stage2_dir.iterdir()), f'Incomplete run: {stage2_dir}'
    subprocess.run(['uv', 'run', 'rogii-train', '--config', 'configs/experiment/pf_trend_blend.yaml', '--run-id', stage2_dir.name, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir)], cwd=repo_dir, check=True)
stage3_dir = artifact_dir / 'stage3_residual_hgb_full_v001'
if not (stage3_dir / 'metrics.json').is_file():
    assert not stage3_dir.exists() or not any(stage3_dir.iterdir()), f'Incomplete run: {stage3_dir}'
    subprocess.run(['uv', 'run', 'rogii-residual', '--config', 'configs/experiment/stage3_residual_hgb.yaml', '--base-run', str(stage2_dir), '--run-id', stage3_dir.name, '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir)], cwd=repo_dir, check=True)
assert (stage3_dir / 'model_full_seed_42.pkl').is_file(), 'Stage 3 full model is missing'
assert (stage3_dir / 'model_full_seed_43.pkl').is_file(), 'Stage 3 second full model is missing'

In [ ]:
SUBMISSION_RUN_ID = 'stage4_submission_v001'
submission_dir = drive_root / 'submissions' / SUBMISSION_RUN_ID
if not (submission_dir / 'submission_report.json').is_file():
    assert not submission_dir.exists() or not any(submission_dir.iterdir()), f'Incomplete submission run: {submission_dir}'
    subprocess.run([
        'uv', 'run', 'rogii-submit',
        '--config', 'configs/experiment/submission_stage4.yaml',
        '--stage3-run', str(stage3_dir),
        '--data-dir', str(data_dir),
        '--output-dir', str(submission_dir),
    ], cwd=repo_dir, check=True)
report = json.loads((submission_dir / 'submission_report.json').read_text())
report

In [ ]:
primary = pd.read_csv(submission_dir / 'submission.csv')
secondary = pd.read_csv(submission_dir / 'submission_stage3.csv')
print('Primary Stage 4:', submission_dir / 'submission.csv')
print('Secondary Stage 3:', submission_dir / 'submission_stage3.csv')
display(primary.head())
assert len(primary) == report['n_rows']
assert report['id_order_matches_sample'] and report['duplicate_ids'] == 0 and report['missing_tvt'] == 0

## Upload

In Kaggle, open the competition's **Submit Predictions** page and upload `submission.csv` from `MyDrive/kaggle/rogii/submissions/stage4_submission_v001/`. This is the primary OOF-selected Stage 4 submission. `submission_stage3.csv` is retained as a secondary, highly correlated control submission; do not replace the primary solely because of a small public-leaderboard difference.